# 03 Embedding Analysis

`chunks.parquet` を BAAI/bge-large-en-v1.5 で 1024 次元 embedding 化し、
1) 同一銘柄・同一セクションの年次変化検知
2) UMAP による 2D 投影クラスタリング
3) 銘柄間類似度比較
を行う。

In [ ]:
# Cell 1: imports + embedder ロード
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
import _helpers
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

device = _helpers.get_device()
model = _helpers.load_embedder(device)
print("device:", device, "dim:", model.get_sentence_embedding_dimension())

In [ ]:
# Cell 2: chunks 読み込み
df_chunks = pd.read_parquet(_helpers.CHUNKS_PARQUET)
print("chunks:", len(df_chunks))

In [ ]:
# Cell 3: バッチ encode (normalize_embeddings=True で cos sim = dot product)
vectors = model.encode(
    df_chunks["text"].tolist(),
    batch_size=16,
    show_progress_bar=True,
    normalize_embeddings=True,
)
print("vectors shape:", vectors.shape)

In [ ]:
# Cell 4: embeddings.parquet 保存 (vector 列は list[float])
df_emb = df_chunks[
    ["filing_id", "ticker", "form", "section_key", "chunk_idx", "filing_date"]
].copy()
df_emb["vector"] = list(vectors)
df_emb.to_parquet(_helpers.EMBEDDINGS_PARQUET)
print("saved:", _helpers.EMBEDDINGS_PARQUET, "rows:", len(df_emb))

In [ ]:
# Cell 5: filing × section 単位で平均プーリング → コサイン類似度 (前期比)
def _avg_pool(group):
    return np.mean(np.vstack(group["vector"].values), axis=0)


pooled = (
    df_emb.groupby(["ticker", "form", "section_key", "filing_id", "filing_date"])
    .apply(lambda g: pd.Series({"mean_vec": _avg_pool(g)}))
    .reset_index()
)
pooled = pooled.sort_values(
    ["ticker", "form", "section_key", "filing_date"]
).reset_index(drop=True)

rows = []
for (ticker, form, section), g in pooled.groupby(["ticker", "form", "section_key"]):
    g = g.sort_values("filing_date").reset_index(drop=True)
    for i in range(1, len(g)):
        v_prev = g.loc[i - 1, "mean_vec"]
        v_now = g.loc[i, "mean_vec"]
        cos = float(
            np.dot(v_prev, v_now) / (np.linalg.norm(v_prev) * np.linalg.norm(v_now))
        )
        rows.append(
            {
                "ticker": ticker,
                "form": form,
                "section_key": section,
                "filing_date": g.loc[i, "filing_date"],
                "cos_sim_prev": cos,
                "diff_score": 1.0 - cos,
            }
        )
df_change = pd.DataFrame(rows)
df_change.head()

In [ ]:
# Cell 6: 変化検知の可視化 (Item 1A の前期比 diff 推移)
import plotly.express as px

risk_diff = df_change[df_change["section_key"] == "item_1a"]
fig = px.line(
    risk_diff,
    x="filing_date",
    y="diff_score",
    color="ticker",
    markers=True,
    line_dash="form",
    title="Risk Factors (Item 1A) - cosine distance to previous filing",
)
fig.show()

In [ ]:
# Cell 7: UMAP 2D 投影 (ticker で色分け)
import umap

X = np.vstack(pooled["mean_vec"].values)
reducer = umap.UMAP(n_components=2, metric="cosine", random_state=42)
X2 = reducer.fit_transform(X)
pooled_plot = pooled.copy()
pooled_plot["x"] = X2[:, 0]
pooled_plot["y"] = X2[:, 1]
fig = px.scatter(
    pooled_plot,
    x="x",
    y="y",
    color="ticker",
    symbol="section_key",
    hover_data=["form", "filing_date"],
    title="UMAP projection of filing×section embeddings",
)
fig.show()

In [ ]:
# Cell 8: 類似銘柄 - AAPL 最新 10-K Item 1A vs MSFT/GOOGL 最新 Item 1A
latest_risk = (
    pooled[(pooled["form"] == "10-K") & (pooled["section_key"] == "item_1a")]
    .sort_values("filing_date")
    .groupby("ticker")
    .tail(1)
    .reset_index(drop=True)
)
vecs = {row["ticker"]: row["mean_vec"] for _, row in latest_risk.iterrows()}
import itertools

sim_rows = []
for a, b in itertools.combinations(vecs.keys(), 2):
    cos = float(
        np.dot(vecs[a], vecs[b]) / (np.linalg.norm(vecs[a]) * np.linalg.norm(vecs[b]))
    )
    sim_rows.append({"a": a, "b": b, "cosine": cos})
pd.DataFrame(sim_rows).sort_values("cosine", ascending=False)